In [1]:
import os
os.environ['USER_AGENT'] = 'BA_GPT/1.0 (contact: walrus_z@hotmail.com)'

In [2]:
import os
import sys
import yaml
import types
import pprint
import logging
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat import EVENT_LOGGER_NAME, TRACE_LOGGER_NAME
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.memory.canvas import TextCanvasMemory
from autogen_ext.models.openai import OpenAIChatCompletionClient
from opentelemetry import trace, _events
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.sdk._logs import LoggerProvider
from opentelemetry.sdk._logs.export import BatchLogRecordProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk._events import EventLoggerProvider
from opentelemetry.exporter.otlp.proto.http._log_exporter import OTLPLogExporter
from opentelemetry.instrumentation.openai import OpenAIInstrumentor


# Project import paths
# Get the current directory (which is project/test)
notebook_dir = os.getcwd()

# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

os.environ["AZURE_TRACING_GEN_AI_CONTENT_RECORDING_ENABLED"] = "true"
os.environ["AZURE_SDK_TRACING_IMPLEMENTATION"] = "opentelemetry"

from agents_creator import create_agents_from_config
from rag_helpers import rag_indexer_runner
from model_loader import load_model_client
from agents_tools_helpers import selector_func

In [3]:
logging.basicConfig(level=logging.WARNING)

# For trace logging.
trace_logger = logging.getLogger(TRACE_LOGGER_NAME)
trace_logger.addHandler(logging.StreamHandler())
trace_logger.setLevel(logging.DEBUG)

# For structured message logging, such as low-level messages between agents.
event_logger = logging.getLogger(EVENT_LOGGER_NAME)
event_logger.addHandler(logging.StreamHandler())
event_logger.setLevel(logging.DEBUG)

In [4]:
# Set up resource
resource = Resource(attributes={
    "service.name": "autogen-test-agentchat"
})

# Set up tracer provider
trace.set_tracer_provider(TracerProvider(resource=resource))

# Configure OTLP exporter
otlp_exporter = OTLPSpanExporter(
    endpoint="http://localhost:4318/v1/traces"
)

# Add span processor
trace.get_tracer_provider().add_span_processor(
    BatchSpanProcessor(otlp_exporter)
)

# Set up logger provider
logger_provider = LoggerProvider(resource=resource)
logger_provider.add_log_record_processor(
    BatchLogRecordProcessor(OTLPLogExporter(endpoint="http://localhost:4318/v1/logs"))
)
_events.set_event_logger_provider(EventLoggerProvider(logger_provider))

# Enable OpenAI instrumentation
OpenAIInstrumentor().instrument()

ERROR:opentelemetry.sdk._shared_internal:Exception while exporting Span.
Traceback (most recent call last):
  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 111] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py", line 787, 

In [5]:
rag_vector_memory = await rag_indexer_runner(lib_path + '/rag_config.yaml')

✅ All documents and URLs are already indexed.


In [6]:
# 1. Create the canvas memory and tool:
text_canvas_memory = TextCanvasMemory()
update_file_tool = text_canvas_memory.get_update_file_tool()

# 2. Register the tool as 'canvas.update_file_tool' for YAML import:
canvas = types.SimpleNamespace()
canvas.update_file_tool = update_file_tool
sys.modules['canvas'] = canvas  # This enables the tool import by your agent factory!

In [18]:
model_client = load_model_client("openai")

In [19]:
model_client_no_parallel_tools = load_model_client("openai_npt")

In [21]:
# Helpful sanity prints:
print("raw:", getattr(model_client_no_parallel_tools, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'parallel_tool_calls': False, 'model': 'gpt-4o', 'api_key': 'REPLACE_WITH_YOUR_API_KEY'}
create_args: {'model': 'gpt-4o'}


In [9]:
# Construct the absolute path to project/lib/agents_config.yaml user_proxy agent
AGENTS_CONFIG = os.path.abspath(os.path.join(notebook_dir, '..', 'lib', 'agents_config.yaml'))
with open(AGENTS_CONFIG, "r") as f:
        config_yaml = yaml.safe_load(f)
agent_names = list(config_yaml.keys())

user, agents = await create_agents_from_config(
    yaml_path=AGENTS_CONFIG,
    agent_names= agent_names,
    model_client=model_client,
    model_client_no_parallel_tools=model_client_no_parallel_tools,
    vector_memory=rag_vector_memory,
    canvas_memory=text_canvas_memory,
    input_func=input
)

In [11]:
for _name, _ag in agents.items():
        has_tools = bool((getattr(_ag, "_tools", None) or []))  # just FYI
        client_cls = type(getattr(_ag, "_model_client", None)).__name__
        name_str = str(_name)
        print(f"[agent-client] {name_str:<28} tools={str(has_tools):<5} client={client_cls}")


[agent-client] planning_agent               tools=False client=OpenAIChatCompletionClient
[agent-client] interviewer_agent            tools=True  client=OpenAIChatCompletionClient
[agent-client] summarizer_agent             tools=False client=OpenAIChatCompletionClient
[agent-client] functional_spec_writer_agent tools=True  client=OpenAIChatCompletionClient
[agent-client] functional_spec_checker_agent tools=True  client=OpenAIChatCompletionClient
[agent-client] diagram_creator_agent        tools=True  client=OpenAIChatCompletionClient
[agent-client] mermaid_code_reviewer_agent  tools=True  client=OpenAIChatCompletionClient
[agent-client] use_case_agent               tools=False client=OpenAIChatCompletionClient
[agent-client] glossary_agent               tools=False client=OpenAIChatCompletionClient
[agent-client] terminate_agent              tools=True  client=OpenAIChatCompletionClient


In [ ]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    pprint.pprint(vars(agent))



In [ ]:
for name, agent in agents.items():
    print(f"\n--- Agent: {name} ---")
    print("Tools:")
    for tool in agent._tools:
        print(f"Tool Name: {tool.name}")
        print(f"Description: {tool.description}\n")


In [12]:
selector_prompt = """
Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure the planner agent has assigned tasks before other agents start working.
Only select one agent.
"""

In [13]:
# Order agents (workflow order), 
agent_order = [
    "planning_agent",
    "interviewer_agent",
    "summarizer_agent",
    "functional_spec_writer_agent",
    "functional_spec_checker_agent",
    "diagram_creator_agent",
    "mermaid_code_reviewer_agent",
    "terminate_agent"
]
member_agents = [agents[name] for name in agent_order if name in agents] + [user]

In [14]:
termination = MaxMessageTermination(100000000) | TextMentionTermination("TERMINATE")

# --- Use SelectorGroupChat ---
team = SelectorGroupChat(
    member_agents,
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False,
    model_client_streaming=False,
    selector_func=selector_func,
)

In [15]:
task = "Interview a customer to collect requirements and scope of a new product then write the functional specifications"

In [16]:
runtime = SingleThreadedAgentRuntime(
            tracer_provider=trace.NoOpTracerProvider(),  # Disable telemetry for runtime.
        )
runtime.start()

In [17]:
try:
    await Console(team.run_stream(task=task))
except Exception as e:
    print(f"Error during team run_stream: {e}")

await runtime.stop()

try:
    await model_client.close()
except Exception as e:
    print(f"Error closing model_client: {e}")

---------- TextMessage (user) ----------
Interview a customer to collect requirements and scope of a new product then write the functional specifications
---------- ModelClientStreamingChunkEvent (planning_agent) ----------
To accomplish the task of collecting requirements and writing the functional specifications for a new product, we'll start by gathering functional requirements through an interview process. 

Here's how we will proceed:

1. The interviewer_agent will begin by asking structured questions to the user to gather detailed functional requirements and scope of the product.
2. Once the interviewer_agent completes the interview and confirms **READY**, the summarizer_agent will summarize the gathered information.
3. After summarization, the functional_spec_writer_agent will draft the functional specification document based on the information.
4. The functional_spec_checker_agent will then review the document for completeness.
5. Once confirmed by the checker, the diagram_crea

id='36cd3634-af4f-4cf2-9f39-c97910f4aeb9' source='interviewer_agent' models_usage=RequestUsage(prompt_tokens=0, completion_tokens=0) metadata={} created_at=datetime.datetime(2025, 9, 19, 3, 32, 58, 854126, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='call_rN7uaYqWpFjj6GBmF5TjqEji', arguments='{"speaker_or_records":[{"speaker":"interviewer","message":"QUESTION: Could you provide an overview of the new product we are considering?","subject":"Product Overview","q_index":1},{"speaker":"user","message":"The scope of the project is to design and develop an online ecommerce website for apparels. Customers will be able to use the frontend website for searching the apparels and place the order by making online payment of the order in advance. Web based admin panel/ backend will help admin user i.e. business owner to manage the products, categories, prices and orders placed by the customers.","subject":"Product Overview","q_index":1}],"file_path":"../conversation/interview.jsonl","ses

---------- ToolCallRequestEvent (interviewer_agent) ----------
[FunctionCall(id='call_rN7uaYqWpFjj6GBmF5TjqEji', arguments='{"speaker_or_records":[{"speaker":"interviewer","message":"QUESTION: Could you provide an overview of the new product we are considering?","subject":"Product Overview","q_index":1},{"speaker":"user","message":"The scope of the project is to design and develop an online ecommerce website for apparels. Customers will be able to use the frontend website for searching the apparels and place the order by making online payment of the order in advance. Web based admin panel/ backend will help admin user i.e. business owner to manage the products, categories, prices and orders placed by the customers.","subject":"Product Overview","q_index":1}],"file_path":"../conversation/interview.jsonl","session_id":"sess-20231124120100"}', name='save_conversation_to_jsonl')]


id='5acd8e5c-36b3-44db-8720-664112feaec8' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 19, 3, 32, 58, 859039, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Saved 2 record(s) to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='call_rN7uaYqWpFjj6GBmF5TjqEji', is_error=False)] type='ToolCallExecutionEvent'
id='5acd8e5c-36b3-44db-8720-664112feaec8' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 19, 3, 32, 58, 859039, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='Saved 2 record(s) to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='call_rN7uaYqWpFjj6GBmF5TjqEji', is_error=False)] type='ToolCallExecutionEvent'
DEBUG:autogen_agentchat.events:id='5acd8e5c-36b3-44db-8720-664112feaec8' source='interviewer_agent' models_usage=None metadata={} created_at=datetime.datetime(2025, 9, 19, 3, 32, 5

---------- ToolCallExecutionEvent (interviewer_agent) ----------
[FunctionExecutionResult(content='Saved 2 record(s) to ../conversation/interview.jsonl', name='save_conversation_to_jsonl', call_id='call_rN7uaYqWpFjj6GBmF5TjqEji', is_error=False)]


ERROR:autogen_core:Error processing publish message for interviewer_agent_5dd4fae9-a663-42f3-ad90-7ba7d50ffb5f/5dd4fae9-a663-42f3-ad90-7ba7d50ffb5f
Traceback (most recent call last):
  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_core/_single_threaded_agent_runtime.py", line 606, in _on_message
    return await agent.on_message(
           ^^^^^^^^^^^^^^^^^^^^^^^
    ...<2 lines>...
    )
    ^
  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_core/_base_agent.py", line 119, in on_message
    return await self.on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_sequential_routed_agent.py", line 67, in on_message_impl
    return await super().on_message_impl(message, ctx)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/run/media/jun/Jun_6T/Code/T

Error during team run_stream: BadRequestError: Error code: 400 - {'error': {'message': "Invalid value for 'parallel_tool_calls': 'parallel_tool_calls' is only allowed when 'tools' are specified.", 'type': 'invalid_request_error', 'param': 'parallel_tool_calls', 'code': None}}
Traceback:
Traceback (most recent call last):

  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/teams/_group_chat/_chat_agent_container.py", line 133, in handle_request
    async for msg in self._agent.on_messages_stream(self._message_buffer, ctx.cancellation_token):
    ...<4 lines>...
            await self._log_message(msg)

  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py", line 989, in on_messages_stream
    async for output_event in self._process_model_result(
    ...<19 lines>...
        yield output_event

  File "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/lib/python3.13/site-pack